# 02 - Contrato y reglas de calidad en Python

## Objetivo y preparacion

Implementar un contrato de datos como una especificacion verificable: estructura esperada, finalidad, responsabilidades, dominios, acciones y condiciones de publicacion. Este laboratorio usa Python local con pandas; no requiere un motor externo de validacion ni servicios cloud. Seleccionar el entorno que contiene las dependencias de `requirements.txt` y ejecutar de arriba hacia abajo. Los datos se generan de nuevo, por lo que no es necesario ejecutar el notebook 01.

## Paso 1 - Definir el uso antes de decidir reglas

La finalidad es analitica de ventas por producto, fecha y moneda, no contacto comercial. Por eso email es opcional y sus fallos son advertencias. El identificador, el importe, la fecha y la integridad del producto son requisitos para cargar. El Owner propuesto es Gerencia Comercial; el Steward revisa ambiguedades; el custodio implementa el ETL. Los nombres del laboratorio no constituyen aprobaciones institucionales.

El contrato tiene version y debe cambiar mediante revision documentada, no porque un lote fallo. La estructura fuente exige un conjunto de columnas, no su orden fisico. `normalizar` verifica ese esquema antes de interpretar valores; luego `evaluar` aplica reglas por fila y conserva todos los hallazgos.

La puerta del caso admite como maximo 20% de filas rechazadas y nunca publica un lote vacio ni uno sin filas aptas. Una columna obligatoria ausente bloquea antes del calculo. El 20% es una decision didactica visible, no una recomendacion general de ISO o DAMA. En un sistema contable podria no admitirse ninguna fila rechazada. Advertencias y cobertura de exactitud requieren seguimiento aunque la puerta permita continuar.

### Organizacion del codigo

La puerta de publicacion se define en la siguiente celda, dentro del notebook que explica el contrato. Sus condiciones y casos de borde quedan visibles junto con las pruebas `unittest`. La funcion de salida JSON tambien es local.

El modulo compartido contiene solo el caso de datos, la normalizacion y la evaluacion por fila usados por los cinco notebooks. La decision de publicar y sus pruebas no estan ocultas en ese modulo. Los notebooks de ETL incluyen su propia puerta visible para poder estudiar y ejecutar cada demostracion de forma independiente.

In [ ]:
import json


def puerta_publicacion(total, rechazadas, max_rechazo=0.20):
    """Aplica el umbral del contrato y rechaza lotes vacios o parametros invalidos."""
    if not 0 <= max_rechazo <= 1 or total < 0 or not 0 <= rechazadas <= total:
        raise ValueError('Parametros invalidos para la puerta de publicacion')
    return total > 0 and rechazadas < total and rechazadas / total <= max_rechazo


def json_seguro(objeto):
    return json.dumps(objeto, ensure_ascii=True, indent=2, allow_nan=False)

In [ ]:
import sys
from pathlib import Path

import pandas as pd

candidatos = [Path.cwd(), Path.cwd() / 'notebook' / 'unidad_III']
carpeta = next((ruta for ruta in candidatos if (ruta / 'gobierno_demo.py').exists()), None)
if carpeta is None:
    raise RuntimeError('Ejecutar desde la raiz del repositorio o notebook/unidad_III')
sys.path.insert(0, str(carpeta.resolve()))
import gobierno_demo as demo

contrato = {
    'version': demo.VERSION_CONTRATO,
    'finalidad': 'analitica de ventas por producto, fecha y moneda',
    'owner': 'Gerencia Comercial',
    'steward': 'Steward Comercial',
    'columnas_fuente': sorted(demo.COLUMNAS_FUENTE),
    'max_rechazo': 0.20,
    'dominio_monedas': ['ARS', 'USD', 'EUR'],
    'evidencia_aprobacion': 'simulada; requiere aprobacion real antes de produccion',
}
print(json_seguro(contrato))
crudos, productos, referencia = demo.crear_datos()
assert demo.esquema_valido(crudos)
datos = demo.normalizar(crudos)
aceptadas, rechazadas, metricas, detalle = demo.evaluar(datos, productos, referencia)
display(metricas[['regla', 'dimension', 'accion', 'evaluados', 'fallidos', 'no_evaluables', 'cumplimiento_pct']])
display(detalle)
publicable = puerta_publicacion(len(datos), len(rechazadas), contrato['max_rechazo'])
assert not publicable
assert len(aceptadas) == 4 and len(rechazadas) == 8
print('Lote bloqueado:', not publicable, '| Tasa de rechazo:', len(rechazadas) / len(datos))

## Paso 2 - Probar el contrato, no solamente el dato feliz

La siguiente celda usa `unittest`, parte de Python, para comprobar casos de borde. Un lote vacio no puede aprobarse por tener cero fallos. Un lote completamente rechazado tampoco debe publicarse. El valor exacto del umbral si se admite; valores superiores no. Parametros imposibles deben provocar un error explicito.

La deriva de esquema se prueba eliminando la columna `moneda` en una copia. `normalizar` debe detenerse antes de transformar o cargar. Agregar una columna tambien requeriria revisar el contrato, porque esta version exige coincidencia del conjunto de campos. No se soluciona ignorando silenciosamente nuevas columnas que podrian contener datos personales.

Los importes requieren dos decimales como maximo y deben ser finitos. Una cadena que represente NaN o tres decimales significativos no se convierte a dinero; una cifra valida se expresa en centavos. La conversion valida no prueba exactitud contra la realidad.

Las pruebas de reglas son evidencia tecnica. La aprobacion del contrato debe constar en el procedimiento organizativo del manual, con version, responsable, fecha efectiva, finalidad e impacto para consumidores.

In [ ]:
import unittest


class PruebasContrato(unittest.TestCase):
    def test_limites_publicacion(self):
        self.assertTrue(puerta_publicacion(10, 2))
        self.assertFalse(puerta_publicacion(10, 3))
        self.assertFalse(puerta_publicacion(0, 0))
        self.assertFalse(puerta_publicacion(10, 10))

    def test_parametros_invalidos(self):
        with self.assertRaises(ValueError):
            puerta_publicacion(3, 4)
        with self.assertRaises(ValueError):
            puerta_publicacion(10, 0, max_rechazo=1.1)

    def test_deriva_esquema(self):
        incompleto = crudos.drop(columns=['moneda'])
        self.assertFalse(demo.esquema_valido(incompleto))
        with self.assertRaisesRegex(ValueError, 'Deriva de esquema'):
            demo.normalizar(incompleto)

    def test_importes(self):
        self.assertEqual(demo.centavos('12.34'), 1234)
        self.assertIsNone(demo.centavos('12.345'))
        self.assertIsNone(demo.centavos('NaN'))

    def test_referencia_ambigua(self):
        ambigua = pd.concat([referencia, referencia.iloc[[0]]], ignore_index=True)
        with self.assertRaisesRegex(ValueError, 'referencia'):
            demo.evaluar(datos, productos, ambigua)

resultado_pruebas = unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(PruebasContrato))
assert resultado_pruebas.wasSuccessful()
print('Cinco pruebas del contrato aprobadas sin motores externos.')

## Interpretacion y ejercicios

El lote original se bloquea: 8 de 12 filas tienen fallos de cuarentena. Las cinco pruebas deben terminar en OK. Revisar metricas y detalle antes de concluir: un 100% de cumplimiento en una regla no prueba calidad de todas las dimensiones, y una referencia parcial no permite declarar exacto todo el lote.

1. Disenar una regla nueva de consistencia entre producto y moneda con un dominio aprobado ficticio. Registrar ID, finalidad, denominador, accion y responsable.
2. Proponer un cambio para exigir email en marketing. Explicar que consumidores afecta y como se versiona el contrato. No alterar las reglas para el uso analitico sin esa decision.
3. Agregar una prueba para una columna adicional y otra para una `fila_fuente` duplicada. Deben bloquear antes de publicar.
4. Discutir si el 20% seria admisible para balances contables y definir un objetivo alternativo con justificacion, no como requisito universal.

**Limite del ejemplo:** el contrato JSON documenta las decisiones; las reglas ejecutables estan en el modulo. No se genera automaticamente codigo a partir de todo el JSON. En un proyecto real deben probarse la consistencia entre especificacion, implementacion y version publicada.